# Model Routing

**Level:** Advanced · **Time:** 90 min

In this notebook, we simulate the architecture of an Enterprise Model Router, optimizing for Cost, Latency, and Reliability.

We will cover 4 distinct patterns:
1. **Semantic Routing:** Picking a model tier based on simple keyword intent.
2. **Capability Filtering:** Dynamically dropping models that lack required capabilities (like Vision).
3. **The Model Cascade:** Promoting from a cheap model to a frontier model upon validation failure.
4. **Reliability Fallbacks:** Catching a 429 Rate Limit error and failing over to a secondary provider.

---
## Pattern 1: Semantic Routing

Sometimes you don't need an LLM to decide routing. If the user's intent is simple, route it directly to the cheapest tier.

In [ ]:
def semantic_router(prompt: str):
    print(f"[Router] Analyzing intent for: '{prompt}'")
    
    if "hello" in prompt.lower() or "ping" in prompt.lower():
        print("✅ [Router] Intent: Simple Greeting -> Routing to tier: MICRO (Llama-3-8b)")
        return "Llama-3-8b"
        
    print("✅ [Router] Intent: Complex -> Routing to tier: FRONTIER (GPT-4o)")
    return "GPT-4o"

semantic_router("Hello there, how are you?")
semantic_router("Write a Python script to calculate the Navier-Stokes equations.")


---
## Pattern 2: Capability Filtering

Before picking a model, we must filter out models that physically cannot handle the request (e.g., they lack Vision).

In [ ]:
model_registry = {
    "haiku": {"cost": 0.25, "vision": True},
    "llama-8b": {"cost": 0.10, "vision": False},
    "sonnet": {"cost": 3.00, "vision": True}
}

def capability_filter(requires_vision: bool):
    print(f"\n[Filter] Request requires vision: {requires_vision}")
    
    eligible_models = []
    for name, metadata in model_registry.items():
        if requires_vision and not metadata["vision"]:
            print(f"❌ [Filter] Dropping {name} (Lacks vision capability)")
            continue
        eligible_models.append(name)
        
    print(f"✅ [Filter] Eligible models: {eligible_models}")
    return eligible_models

capability_filter(requires_vision=True)


---
## Pattern 3: The Model Cascade

We try the cheap model first. If it returns an invalid response (fails a programmatic assertion), we promote to the expensive model.

In [ ]:
import json

def mock_llm_call(model, prompt):
    print(f"  [API] Calling {model}...")
    if model == "haiku":
        return "Here is the JSON: { bad_json: true" # Simulated failure
    return '{"status": "success", "data": 123}' # Simulated success

def model_cascade(prompt):
    print("\n[Cascade] Step 1: Trying cheap model (Haiku)")
    response_1 = mock_llm_call("haiku", prompt)
    
    try:
        # The Programmatic Assertion
        json.loads(response_1)
        print("✅ [Cascade] Assertion Passed. Returning cheap model response.")
        return response_1
    except json.JSONDecodeError:
        print("🚨 [Cascade] Assertion Failed! (Invalid JSON). Promoting to Frontier model.")
        
    print("\n[Cascade] Step 2: Trying frontier model (Sonnet)")
    response_2 = mock_llm_call("sonnet", prompt)
    print("✅ [Cascade] Frontier model succeeded.")
    return response_2

model_cascade("Return a JSON object.")


---
## Pattern 4: Reliability Fallbacks

If Provider A experiences an outage or rate limit, we must transparently failover to Provider B.

In [ ]:
class RateLimitError(Exception):
    pass

def mock_provider_a_call():
    print("  [API] Calling Provider A (OpenAI)...")
    raise RateLimitError("429 Too Many Requests")

def mock_provider_b_call():
    print("  [API] Calling Provider B (Anthropic)...")
    return "Response from Provider B"

def reliability_router():
    print("\n[Router] Executing Primary Route...")
    try:
        response = mock_provider_a_call()
        return response
    except RateLimitError as e:
        print(f"🚨 [Router] Caught error: {e}")
        print("[Router] Initiating Failover to Secondary Provider...")
        response = mock_provider_b_call()
        print("✅ [Router] Failover successful.")
        return response

reliability_router()
